In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve, roc_auc_score


# Import finalized feature data


In [ ]:
data_file = "../processed_data/finalized_feature_data_2024-03-01_2026-05-02.csv"
feature_data = (
    pd.read_csv(data_file, parse_dates=["timestamp_utc"])
    .set_index("timestamp_utc")
    .sort_index()
)

feature_data["Day_of_week_sin"] = np.sin(2 * np.pi * feature_data["Day_of_week"] / 7)
feature_data["Day_of_week_cos"] = np.cos(2 * np.pi * feature_data["Day_of_week"] / 7)
feature_data["Month_sin"] = np.sin(2 * np.pi * feature_data["Month"] / 12)
feature_data["Month_cos"] = np.cos(2 * np.pi * feature_data["Month"] / 12)

observed_cols = [column for column in feature_data.columns if column.startswith("observed_")]
forecast_cols = [column for column in feature_data.columns if column.startswith("forecast_")]

feature_combined = feature_data.drop(columns=forecast_cols).copy()
forecast_data = feature_data[forecast_cols].copy()

display(feature_data.shape)
display(feature_combined.shape)
display(forecast_data.shape)
display(feature_combined.columns)
display(forecast_data.columns)


In [ ]:
# Basic timestamp checks.
for name, df in {"feature_combined": feature_combined, "forecast_data": forecast_data}.items():
    if df.index.has_duplicates:
        raise ValueError(f"{name} has duplicated timestamps")

    hourly_steps = df.index.to_series().sort_values().diff().dropna()
    if not hourly_steps.eq(pd.Timedelta(hours=1)).all():
        raise ValueError(f"{name} is not continuous hourly data")


## Models:

All three models predict whether the electricity price is negative at target hour `t`. SMARD and price inputs must only use information available at least 24 hours before the target hour, so generation and price history should be taken from `t-24` and earlier. Calendar features for the target hour are allowed because hour, weekday, month, and weekend status are known ahead of time.

1. **Past-only baseline**: use only historical sequence information ending at `t-24`. This includes SMARD/price history and, if weather is included, only observed weather up to `t-24`. This model measures how much signal is available without target-day weather information.

2. **Forecast-weather model**: use the same SMARD/price cutoff as the baseline, but add Open-Meteo Previous Runs day-1 weather forecasts aligned to the target weather valid times. Rows with `timestamp_utc == t` represent forecasts for hour `t` that were available about 24 hours earlier, so they are deployable for day-ahead prediction.

3. **Oracle real-weather model**: use the same SMARD/price cutoff, but replace forecast weather with observed weather for the target weather window. This model is not deployable, but it estimates the upper bound from perfect weather information and helps measure the cost of forecast error.

We will call these three models `past_model`, `forecast_model`, and `oracle_model`

In [ ]:
# Config
lookback = 24
valid_start = pd.Timestamp("2025-07-01", tz="UTC")
test_start = pd.Timestamp("2025-12-01", tz="UTC")

# TCN capacity. Channels are the hidden feature count in each temporal block.
tcn_channels = (64, 64, 64, 64)
kernel_size = 3
dropout = 0.1

learning_rate = 0.001
batch_size = 64
epochs = 10
patience = 5


In [ ]:
# Model input tables. Target time is t; SMARD and price history end at t-24.
target = feature_combined["Negative_price"].astype(int).sort_index()

calendar_cols = [
    "Hour",
    "Day_of_week",
    "Month",
    "Is_weekend",
    "Hour_sin",
    "Hour_cos",
    "Day_of_week_sin",
    "Day_of_week_cos",
    "Month_sin",
    "Month_cos",
]

observed_cols = [c for c in feature_combined.columns if c.startswith("observed_")]
forecast_cols = [c for c in forecast_data.columns if c.startswith("forecast_")]
smard_input_cols = [
    c for c in feature_combined.columns
    if c not in calendar_cols + ["Negative_price"] + observed_cols
]

observed_weather = feature_combined[observed_cols].rename(columns=lambda c: c.replace("observed_", ""))
forecast_weather = forecast_data[forecast_cols].rename(columns=lambda c: c.replace("forecast_", ""))

if sorted(observed_weather.columns) != sorted(forecast_weather.columns):
    raise ValueError("Observed and forecast weather columns do not match after removing prefixes")

weather_cols = observed_weather.columns.tolist()
feature_cols = calendar_cols + smard_input_cols + weather_cols

past_raw = pd.concat(
    [feature_combined[calendar_cols], feature_combined[smard_input_cols], observed_weather],
    axis=1,
).sort_index()

# Future rows only use known calendar features and weather. Future SMARD/price columns become NaN placeholders.
forecast_future_raw = (
    feature_combined[calendar_cols]
    .join(forecast_weather, how="inner")
    .reindex(columns=feature_cols)
    .sort_index()
)

oracle_future_raw = (
    feature_combined[calendar_cols]
    .join(observed_weather, how="inner")
    .reindex(columns=feature_cols)
    .sort_index()
)

pd.DataFrame({
    "table": ["past_raw", "forecast_future_raw", "oracle_future_raw"],
    "shape": [past_raw.shape, forecast_future_raw.shape, oracle_future_raw.shape],
    "start": [past_raw.index.min(), forecast_future_raw.index.min(), oracle_future_raw.index.min()],
    "end": [past_raw.index.max(), forecast_future_raw.index.max(), oracle_future_raw.index.max()],
})


In [ ]:
# Scale with training-period statistics only.
train_rows = past_raw[past_raw.index < valid_start].reindex(columns=feature_cols)
feature_mean = train_rows.mean()
feature_std = train_rows.std().replace(0, 1).fillna(1)

def scale_features(df):
    return ((df.reindex(columns=feature_cols) - feature_mean) / feature_std).fillna(0.0)

past_scaled = scale_features(past_raw)
forecast_future_scaled = scale_features(forecast_future_raw)
oracle_future_scaled = scale_features(oracle_future_raw)


In [ ]:
# Build sequences. Past part is t-(24+lookback-1)...t-24.
# If future_X is passed, append the deployable weather path t-23...t.
def build_sequences(X, y, future_X=None, lookback=24):
    Xs, ys, times = [], [], []

    X_values = X.to_numpy(dtype=np.float32)
    X_start = X.index.min()

    if future_X is not None:
        future_values = future_X.to_numpy(dtype=np.float32)
        future_start_time = future_X.index.min()

    for t in y.index:
        past_start = t - pd.Timedelta(hours=24 + lookback - 1)
        past_end = t - pd.Timedelta(hours=24)

        start_pos = int((past_start - X_start).total_seconds() // 3600)
        end_pos = int((past_end - X_start).total_seconds() // 3600) + 1

        if start_pos < 0 or end_pos > len(X_values):
            continue

        seq = X_values[start_pos:end_pos]
        if len(seq) != lookback:
            continue

        if future_X is not None:
            future_start = t - pd.Timedelta(hours=23)
            future_start_pos = int((future_start - future_start_time).total_seconds() // 3600)
            future_end_pos = int((t - future_start_time).total_seconds() // 3600) + 1

            if future_start_pos < 0 or future_end_pos > len(future_values):
                continue

            future_seq = future_values[future_start_pos:future_end_pos]
            if len(future_seq) != 24:
                continue

            seq = np.vstack([seq, future_seq])

        Xs.append(seq)
        ys.append(y.loc[t])
        times.append(t)

    return np.array(Xs), np.array(ys, dtype=np.float32), pd.DatetimeIndex(times)


In [ ]:
# Build aligned datasets for one lookback value.
def split_sequences(X, y, times):
    train_mask = times < valid_start
    valid_mask = (times >= valid_start) & (times < test_start)
    test_mask = times >= test_start

    return {
        "X_train": X[train_mask],
        "y_train": y[train_mask],
        "X_valid": X[valid_mask],
        "y_valid": y[valid_mask],
        "X_test": X[test_mask],
        "y_test": y[test_mask],
    }


def keep_common_times(X, y, times, common_times):
    mask = times.isin(common_times)
    return X[mask], y[mask], times[mask]


def build_model_datasets(current_lookback):
    X_past, y_past, times_past = build_sequences(past_scaled, target, lookback=current_lookback)
    X_forecast, y_forecast, times_forecast = build_sequences(
        past_scaled, target, future_X=forecast_future_scaled, lookback=current_lookback
    )
    X_oracle, y_oracle, times_oracle = build_sequences(
        past_scaled, target, future_X=oracle_future_scaled, lookback=current_lookback
    )

    common_times = times_past.intersection(times_forecast).intersection(times_oracle).sort_values()

    X_past, y_past, times_past = keep_common_times(X_past, y_past, times_past, common_times)
    X_forecast, y_forecast, times_forecast = keep_common_times(X_forecast, y_forecast, times_forecast, common_times)
    X_oracle, y_oracle, times_oracle = keep_common_times(X_oracle, y_oracle, times_oracle, common_times)

    datasets = {
        "past_model": split_sequences(X_past, y_past, times_past),
        "forecast_model": split_sequences(X_forecast, y_forecast, times_forecast),
        "oracle_model": split_sequences(X_oracle, y_oracle, times_oracle),
    }

    summary = pd.DataFrame([
        {"model": "past_model", "lookback": current_lookback, "train": len(datasets["past_model"]["y_train"]), "valid": len(datasets["past_model"]["y_valid"]), "test": len(datasets["past_model"]["y_test"]), "sequence_shape": X_past.shape[1:]},
        {"model": "forecast_model", "lookback": current_lookback, "train": len(datasets["forecast_model"]["y_train"]), "valid": len(datasets["forecast_model"]["y_valid"]), "test": len(datasets["forecast_model"]["y_test"]), "sequence_shape": X_forecast.shape[1:]},
        {"model": "oracle_model", "lookback": current_lookback, "train": len(datasets["oracle_model"]["y_train"]), "valid": len(datasets["oracle_model"]["y_valid"]), "test": len(datasets["oracle_model"]["y_test"]), "sequence_shape": X_oracle.shape[1:]},
    ]).set_index("model")

    return datasets, summary, common_times


datasets, dataset_summary, common_times = build_model_datasets(lookback)
display(dataset_summary)
print("time span:", common_times.min(), "to", common_times.max())


## TCN model


In [ ]:
# TCN structure
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        if self.chomp_size == 0:
            return x
        return x[:, :, :-self.chomp_size]


class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, dropout=0.1):
        super().__init__()
        padding = (kernel_size - 1) * dilation

        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        if in_channels != out_channels:
            self.downsample = nn.Conv1d(in_channels, out_channels, kernel_size=1)
        else:
            self.downsample = nn.Identity()

        self.final_relu = nn.ReLU()

    def forward(self, x):
        out = self.conv1(x)
        out = self.chomp1(out)
        out = self.relu1(out)
        out = self.dropout1(out)

        out = self.conv2(out)
        out = self.chomp2(out)
        out = self.relu2(out)
        out = self.dropout2(out)

        return self.final_relu(out + self.downsample(x))


class CurtailmentTCN(nn.Module):
    def __init__(self, input_dim, channels=(64, 64, 64, 64), kernel_size=3, dropout=0.1):
        super().__init__()

        layers = []
        in_channels = input_dim
        for i, out_channels in enumerate(channels):
            layers.append(
                TemporalBlock(
                    in_channels,
                    out_channels,
                    kernel_size=kernel_size,
                    dilation=2 ** i,
                    dropout=dropout,
                )
            )
            in_channels = out_channels

        self.tcn = nn.Sequential(*layers)
        self.fc = nn.Linear(channels[-1], 1)

    def forward(self, x):
        # Conv1d expects batch, features, time.
        x = x.transpose(1, 2)
        tcn_out = self.tcn(x)
        last_step = tcn_out[:, :, -1]
        return self.fc(last_step).squeeze(-1)


In [ ]:
# Dataloaders
def make_loader(X, y, shuffle):
    dataset = TensorDataset(
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


In [ ]:
# Training and evaluation helpers
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    losses = []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    return float(np.mean(losses))


def evaluate_loss(model, loader, criterion, device):
    model.eval()
    losses = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            losses.append(loss.item())

    return float(np.mean(losses))


def predict_probabilities(model, loader, device):
    model.eval()
    probabilities, targets = [], []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            logits = model(X_batch.to(device))
            probabilities.append(torch.sigmoid(logits).cpu().numpy())
            targets.append(y_batch.numpy())

    return np.concatenate(probabilities), np.concatenate(targets)


def threshold_sweep(y_true, probabilities):
    thresholds = np.linspace(0.05, 0.95, 91)
    f1_scores = [f1_score(y_true, probabilities >= t, zero_division=0) for t in thresholds]
    best_index = int(np.argmax(f1_scores))
    return thresholds[best_index], f1_scores[best_index]


def plot_threshold_and_pr_curve(y_true, probabilities, threshold=None, title="TCN validation"):
    precision, recall, pr_thresholds = precision_recall_curve(y_true, probabilities)
    f1_scores = 2 * precision * recall / (precision + recall + 1e-10)
    best_index = int(np.argmax(f1_scores[:-1]))
    best_threshold = pr_thresholds[best_index]
    best_f1 = f1_scores[best_index]
    selected_threshold = best_threshold if threshold is None else threshold
    pr_auc = average_precision_score(y_true, probabilities)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(pr_thresholds, f1_scores[:-1])
    axes[0].axvline(selected_threshold, color="red", linestyle="--", label=f"threshold = {selected_threshold:.3f}")
    axes[0].set(xlabel="Probability threshold", ylabel="F1 score", title=f"{title}: F1 vs threshold")
    axes[0].legend()
    axes[0].grid(alpha=0.2)

    axes[1].plot(recall, precision, label=f"PR-AUC = {pr_auc:.3f}")
    axes[1].axhline(np.mean(y_true), color="gray", linestyle="--", label="Positive-rate baseline")
    axes[1].set(xlabel="Recall", ylabel="Precision", title=f"{title}: PR curve")
    axes[1].legend()
    axes[1].grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

    return {"best_threshold": best_threshold, "best_f1": best_f1, "pr_auc": pr_auc}


In [ ]:
# Run one model. The checkpoint is selected by valid PR-AUC.
from torch.utils.data.dataloader import DataLoader


from torch._tensor import Tensor


def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def set_random_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def run_experiment(
    name,
    data,
    channels=tcn_channels,
    kernel_size=kernel_size,
    dropout=dropout,
    seed=42,
    lookback_value=None,
    return_predictions=False,
    verbose=True,
):
    set_random_seed(seed)
    device = get_device()
    if verbose:
        print(f"Using device : {device}; seed={seed}")

    input_dim = data["X_train"].shape[2]
    model = CurtailmentTCN(input_dim, channels=channels, kernel_size=kernel_size, dropout=dropout).to(device)

    num_pos = data["y_train"].sum()
    num_neg = len(data["y_train"]) - num_pos
    pos_weight = torch.tensor([num_neg / max(num_pos, 1)], dtype=torch.float32).to(device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    train_loader = make_loader(data["X_train"], data["y_train"], shuffle=True)
    valid_loader = make_loader(data["X_valid"], data["y_valid"], shuffle=False)

    best_state = None
    best_epoch = 0
    best_valid_pr_auc = -np.inf
    best_valid_f1_seen = -np.inf
    best_valid_f1_threshold = 0.5
    epochs_without_improvement = 0

    for epoch in range(epochs):
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
        valid_loss = evaluate_loss(model, valid_loader, criterion, device)
        valid_prob, valid_true = predict_probabilities(model, valid_loader, device)
        valid_pr_auc = average_precision_score(valid_true, valid_prob)
        threshold, valid_f1 = threshold_sweep(valid_true, valid_prob)

        if valid_f1 > best_valid_f1_seen:
            best_valid_f1_seen = valid_f1
            best_valid_f1_threshold = threshold

        if valid_pr_auc > best_valid_pr_auc:
            best_valid_pr_auc = valid_pr_auc
            best_epoch = epoch + 1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if verbose:
            print(
                f"{name} epoch {epoch + 1:02d}: "
                f"train_loss={train_loss:.4f}, "
                f"valid_loss={valid_loss:.4f}, "
                f"valid_F1={valid_f1:.4f}, "
                f"threshold={threshold:.2f}, "
                f"valid_PR_AUC={valid_pr_auc:.4f}"
            )

        if epochs_without_improvement >= patience:
            if verbose:
                print(f"Early stopping at epoch {epoch + 1:02d}; best valid_PR_AUC={best_valid_pr_auc:.4f}")
            break

    model.load_state_dict(best_state)

    valid_prob, valid_true = predict_probabilities(model, valid_loader, device)
    selected_threshold, selected_valid_f1 = threshold_sweep(valid_true, valid_prob)

    result = {
        "model": name,
        "seed": seed,
        "lookback": lookback if lookback_value is None else lookback_value,
        "channels": str(tuple(channels)),
        "kernel_size": kernel_size,
        "dropout": dropout,
        "best_epoch_by_PR_AUC": best_epoch,
        "train_size": len(data["y_train"]),
        "valid_size": len(data["y_valid"]),
        "sequence_shape": str(data["X_train"].shape[1:]),
        "valid_PR_AUC": average_precision_score(valid_true, valid_prob),
        "selected_valid_F1": selected_valid_f1,
        "selected_threshold": selected_threshold,
        "best_valid_F1_seen": best_valid_f1_seen,
        "best_valid_F1_threshold": best_valid_f1_threshold,
 }
    if return_predictions:
        result["valid_true"] = valid_true
        result["valid_prob"] = valid_prob

    return result


In [ ]:
# Run all three models.
data_by_name = datasets

results = [
    run_experiment("past_model", data_by_name["past_model"], seed=42, lookback_value=lookback),
    run_experiment("forecast_model", data_by_name["forecast_model"], seed=42, lookback_value=lookback),
    run_experiment("oracle_model", data_by_name["oracle_model"], seed=42, lookback_value=lookback),
]

results_df = pd.DataFrame(results).set_index("model")
display(results_df)


## Tuning


In [ ]:
tuning_seeds = [1, 2, 3]
lookback_values = [24, 48, 72, 168]
channel_options = [
    (32, 32, 32, 32),
    (64, 64, 64, 64),
    (32, 32, 64, 64),
]

tuning_rows = []
tuning_predictions = {}

for current_lookback in lookback_values:
    lookback = current_lookback
    datasets, dataset_summary, common_times = build_model_datasets(lookback)
    print("lookback =", lookback)
    # display(dataset_summary)

    for current_channels in channel_options:
        combo_rows = []
        print(f"  tuning channels={current_channels} over seeds={tuning_seeds}")

        for seed in tuning_seeds:
            result = run_experiment(
                "forecast_model",
                datasets["forecast_model"],
                channels=current_channels,
                kernel_size=kernel_size,
                dropout=dropout,
                seed=seed,
                lookback_value=current_lookback,
                return_predictions=True,
                verbose=False,
            )
            prediction_key = (current_lookback, str(tuple(current_channels)), seed)
            tuning_predictions[prediction_key] = {
                "valid_true": result.pop("valid_true"),
                "valid_prob": result.pop("valid_prob"),
            }
            tuning_rows.append(result)
            combo_rows.append(result)

        combo_df = pd.DataFrame(combo_rows)
        print(
            "    3-seed mean: "
            f"selected_valid_F1={combo_df['selected_valid_F1'].mean():.4f} "
            f"(+/- {combo_df['selected_valid_F1'].std():.4f}), "
            f"valid_PR_AUC={combo_df['valid_PR_AUC'].mean():.4f} "
            f"(+/- {combo_df['valid_PR_AUC'].std():.4f})"
        )


tuning_seed_results_df = pd.DataFrame(tuning_rows)

tuning_summary_df = (
    tuning_seed_results_df
    .groupby(["lookback", "channels", "kernel_size", "dropout", "sequence_shape"], sort=False)
    .agg(
        valid_PR_AUC=("valid_PR_AUC", "mean"),
        selected_valid_F1=("selected_valid_F1", "mean"),
        selected_threshold_mean=("selected_threshold", "mean"),
        valid_PR_AUC_std=("valid_PR_AUC", "std"),
        selected_valid_F1_std=("selected_valid_F1", "std"),
    )
    .reset_index()
    .sort_values(["selected_valid_F1", "valid_PR_AUC"], ascending=False)
    .reset_index(drop=True)
)

best_tuning_row = tuning_summary_df.iloc[0]
best_lookback = int(best_tuning_row["lookback"])
best_channels = tuple(int(part.strip()) for part in best_tuning_row["channels"].strip("()").split(",") if part.strip())
best_validation_threshold = float(best_tuning_row["selected_threshold_mean"])
best_final_epochs = max(1, int(round(best_tuning_row["best_epoch_by_PR_AUC_mean"])))

print("Best tuning parameters selected by 3-seed mean selected_valid_F1")
print({
    "lookback": best_lookback,
    "channels": best_channels,
    "selected_valid_F1": best_tuning_row["selected_valid_F1"],
    "valid_PR_AUC": best_tuning_row["valid_PR_AUC"],
    "validation_threshold_mean": best_validation_threshold,
})

display(tuning_summary_df)


## Best Tuning Diagnostics


In [ ]:
plot_tuning = tuning_summary_df.copy()
plot_tuning["label"] = "lb" + plot_tuning["lookback"].astype(str) + " " + plot_tuning["channels"]
plot_tuning = plot_tuning.sort_values("selected_valid_F1", ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(15, max(5, 0.45 * len(plot_tuning))))
axes[0].barh(plot_tuning["label"], plot_tuning["selected_valid_F1"], xerr=plot_tuning["selected_valid_F1_std"].fillna(0))
axes[0].set(xlabel="3-seed mean selected validation F1", title="TCN tuning by F1")
axes[0].grid(axis="x", alpha=0.25)

axes[1].barh(plot_tuning["label"], plot_tuning["valid_PR_AUC"], xerr=plot_tuning["valid_PR_AUC_std"].fillna(0))
axes[1].set(xlabel="3-seed mean validation PR-AUC", title="TCN tuning by PR-AUC")
axes[1].grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

best_pr_auc_row = tuning_summary_df.sort_values(["valid_PR_AUC", "selected_valid_F1"], ascending=False).iloc[0]
diagnostic_lookback = int(best_pr_auc_row["lookback"])
diagnostic_channels = tuple(int(part.strip()) for part in best_pr_auc_row["channels"].strip("()").split(",") if part.strip())
diagnostic_seed = 1
diagnostic_seed_row = tuning_seed_results_df[
    (tuning_seed_results_df["lookback"] == diagnostic_lookback)
    & (tuning_seed_results_df["channels"] == str(tuple(diagnostic_channels)))
    & (tuning_seed_results_df["seed"] == diagnostic_seed)
].iloc[0]
best_prediction_key = (diagnostic_lookback, str(tuple(diagnostic_channels)), diagnostic_seed)
best_prediction = tuning_predictions[best_prediction_key]

print("Diagnostic combo selected by 3-seed mean valid_PR_AUC")
display(pd.DataFrame([best_pr_auc_row]))
print(f"Plotting validation curves for seed={diagnostic_seed}")
validation_curve_summary = plot_threshold_and_pr_curve(
    best_prediction["valid_true"],
    best_prediction["valid_prob"],
    threshold=float(diagnostic_seed_row["selected_threshold"]),
    title=f"forecast_model TCN lb{diagnostic_lookback} seed{diagnostic_seed}",
)
display(pd.DataFrame([validation_curve_summary]))


## Final Train+Validation Fit and Test Evaluation


In [ ]:
def train_final_forecast_model(data, channels, seed, final_epochs, threshold):
    set_random_seed(seed)
    device = get_device()
    X_dev = np.concatenate([data["X_train"], data["X_valid"]], axis=0)
    y_dev = np.concatenate([data["y_train"], data["y_valid"]], axis=0)

    input_dim = X_dev.shape[2]
    model = CurtailmentTCN(input_dim, channels=channels, kernel_size=kernel_size, dropout=dropout).to(device)

    num_pos = y_dev.sum()
    num_neg = len(y_dev) - num_pos
    pos_weight = torch.tensor([num_neg / max(num_pos, 1)], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    dev_loader = make_loader(X_dev, y_dev, shuffle=True)
    test_loader = make_loader(data["X_test"], data["y_test"], shuffle=False)

    train_losses = []
    for epoch in range(final_epochs):
        train_loss = train_epoch(model, dev_loader, criterion, optimizer, device)
        train_losses.append(train_loss)
        print(f"final seed={seed} epoch {epoch + 1:02d}/{final_epochs}: train_loss={train_loss:.4f}")

    test_prob, test_true = predict_probabilities(model, test_loader, device)
    return {
        "model": "forecast_model_final",
        "seed": seed,
        "lookback": best_lookback,
        "channels": str(tuple(channels)),
        "final_epochs": final_epochs,
        "threshold_from_validation": threshold,
        "train_valid_size": len(y_dev),
        "test_size": len(test_true),
        "test_PR_AUC": average_precision_score(test_true, test_prob),
        "test_ROC_AUC": roc_auc_score(test_true, test_prob),
        "test_F1_at_validation_threshold": f1_score(test_true, test_prob >= threshold, zero_division=0),
        "test_positive_rate": test_true.mean(),
        "test_true": test_true,
        "test_prob": test_prob,
        "train_loss_last": train_losses[-1],
    }


lookback = best_lookback
datasets, dataset_summary, common_times = build_model_datasets(best_lookback)
final_forecast_data = datasets["forecast_model"]

final_results = []
final_predictions = {}
for seed in tuning_seeds:
    result = train_final_forecast_model(
        final_forecast_data,
        channels=best_channels,
        seed=seed,
        final_epochs=best_final_epochs,
        threshold=best_validation_threshold,
    )
    final_predictions[seed] = {
        "test_true": result.pop("test_true"),
        "test_prob": result.pop("test_prob"),
    }
    final_results.append(result)

final_test_results_df = pd.DataFrame(final_results)
final_test_summary = final_test_results_df.agg({
    "test_PR_AUC": ["mean", "std"],
    "test_ROC_AUC": ["mean", "std"],
    "test_F1_at_validation_threshold": ["mean", "std"],
    "test_positive_rate": ["mean"],
}).T

print("Final forecast_model test results after training on train+validation")
display(final_test_results_df)
display(final_test_summary)

best_final_seed = int(final_test_results_df.sort_values("test_F1_at_validation_threshold", ascending=False).iloc[0]["seed"])
best_final_prediction = final_predictions[best_final_seed]
print(f"Plotting test curves for final model seed={best_final_seed}")
test_curve_summary = plot_threshold_and_pr_curve(
    best_final_prediction["test_true"],
    best_final_prediction["test_prob"],
    threshold=best_validation_threshold,
    title=f"final forecast_model TCN test seed{best_final_seed}",
)
display(pd.DataFrame([test_curve_summary]))
